#### transformation of dim_table

In [50]:
# read csv code and converting into data frame
df = spark.read.format('csv').options(header='True', inferSchema=True).load('abfss://atliq_motor_space@onelake.dfs.fabric.microsoft.com/atliq_motor__lakehouse.Lakehouse/Files/Bronze/dim_date.csv')
display(df)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 257c4304-3808-421c-8088-cf53855430ed)

In [51]:
from pyspark.sql.functions import to_date

df_silver = df.withColumn('date', to_date('date', 'dd-MMM-yy'))
display(df_silver)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a19d7860-22f1-4a3f-adfb-a8e0eaf69dfd)

In [52]:
df_silver.write.format('delta').mode('overwrite').saveAsTable('silver_dim_table')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 54, Finished, Available, Finished, False)

#### transformation of electric vehicle sales by makers

In [53]:
df = spark.read.format('csv').options(header='True', inferSchema=True).load('abfss://atliq_motor_space@onelake.dfs.fabric.microsoft.com/atliq_motor__lakehouse.Lakehouse/Files/Bronze/electric_vehicle_sales_by_makers.csv')
display(df)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 95ea6819-9784-4a9c-abea-3eb2c4676c42)

In [5]:
df.createOrReplaceTempView('sampledata')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 7, Finished, Available, Finished, False)

In [54]:
from pyspark.sql.functions import to_date

df_silver2 = df.withColumn('date', to_date('date', 'dd-MMM-yy'))
display(df_silver2)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46ebd01c-6b7e-43dd-b150-a0391a812c16)

In [7]:
df_silver2.createOrReplaceTempView('sampledata2')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 9, Finished, Available, Finished, False)

In [8]:
%%sql
select count(*) from sampledata2 where electric_vehicles_sold=0

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [55]:
display(df_silver2)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 23d18936-8bc8-4eef-a665-417089c74b61)

In [57]:
from pyspark.sql.functions import regexp_replace
df_silver4 = df.withColumn('maker', regexp_replace('maker', 'OLA ELECTRIC', 'OLA Electric')) \
            .withColumn('maker', regexp_replace('maker', 'Mercedes -Benz AG', 'Mercedes-Benz AG'))\
            .withColumn('maker', regexp_replace('maker', 'Mahindra & Mahindra', 'Mahindra'))

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 59, Finished, Available, Finished, False)

In [58]:
display(df_silver4)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 33ccad43-b845-4899-964f-d3d887d9cd01)

In [59]:
from pyspark.sql.functions import to_date

df_silver4 = df.withColumn('date', to_date('date', 'dd-MMM-yy'))
display(df_silver4)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 31d29a3c-fe70-438d-9176-cc2556f73d36)

In [61]:
df_silver4.write.format('delta').mode('overwrite').saveAsTable('silver_electric_vehicle_sales_by_makers')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 63, Finished, Available, Finished, False)

#### transformation of electric_vehicle_sales_by_state

In [13]:
df = spark.read.format('csv').options(header='True', inferSchema=True).load('abfss://atliq_motor_space@onelake.dfs.fabric.microsoft.com/atliq_motor__lakehouse.Lakehouse/Files/Bronze/electric_vehicle_sales_by_state.csv')
display(df)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c84f172-e2a1-4b5c-b2c4-1959352d915c)

In [14]:
from pyspark.sql.functions import to_date

df_silver3 = df.withColumn('date', to_date('date', 'dd-MMM-yy'))
display(df_silver3)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7274fda7-2943-4bbf-b9a9-22a2e1ec0bf9)

In [15]:
df_silver3.write.format('delta').mode('overwrite').saveAsTable('silver_electric_vehicle_sales_by_state')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 17, Finished, Available, Finished, False)

## **For Gold Layer**

In [22]:
#Ev Sales by state and year
from pyspark.sql import functions as F

df_state = spark.table('silver_electric_vehicle_sales_by_state')
df_date = spark.table('silver_dim_table')

gold_state = df_state \
    .join(df_date, 'date', 'inner') \
    .groupBy('fiscal_year', 'quarter', 'state', 'vehicle_category') \
    .agg(
        F.sum('electric_vehicles_sold').alias('total_ev_sold'),
        F.sum('total_vehicles_sold').alias('total_vehicles'),
        F.round(F.sum('electric_vehicles_sold') / F.sum('total_vehicles_sold') * 100, 2).alias('ev_penetration_rate')
    )

gold_state.write.format('delta').mode('overwrite').saveAsTable('gold_ev_sales_by_state')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 24, Finished, Available, Finished, False)

In [62]:
#EV Sales by Maker & Year
from pyspark.sql import functions as F

df_maker = spark.table('silver_electric_vehicle_sales_by_makers')
df_date = spark.table('silver_dim_table')

gold_maker = df_maker \
    .join(df_date, 'date', 'inner') \
    .groupBy('fiscal_year', 'quarter', 'maker', 'vehicle_category') \
    .agg(
        F.sum('electric_vehicles_sold').alias('total_ev_sold')
    )

print("Gold rows:", gold_maker.count())
display(gold_maker)

gold_maker.write.format('delta').mode('overwrite').saveAsTable('gold_ev_sales_by_maker')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 64, Finished, Available, Finished, False)

Gold rows: 272


SynapseWidget(Synapse.DataFrame, 301e9907-bd07-4b80-8f33-c830fac9ef53)

In [63]:
df_maker.select('date').show(3)
df_date.select('date').show(3)

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 65, Finished, Available, Finished, False)

+----------+
|      date|
+----------+
|2021-04-01|
|2022-04-01|
|2021-05-01|
+----------+
only showing top 3 rows

+----------+
|      date|
+----------+
|2021-04-01|
|2021-05-01|
|2021-06-01|
+----------+
only showing top 3 rows



In [64]:
# Overall EV Trend (for KPI cards)
from pyspark.sql import functions as F

gold_trend = df_state \
    .join(df_date, 'date', 'inner') \
    .groupBy('fiscal_year', 'quarter') \
    .agg(
        F.sum('electric_vehicles_sold').alias('total_ev_sold'),
        F.sum('total_vehicles_sold').alias('total_vehicles'),
        F.round(F.sum('electric_vehicles_sold') / F.sum('total_vehicles_sold') * 100, 2).alias('ev_penetration_rate')
    )

gold_trend.write.format('delta').mode('overwrite').saveAsTable('gold_ev_overall_trend')

StatementMeta(, 1ffd9cbe-57ae-4022-ac20-d9673ad8de4b, 66, Finished, Available, Finished, False)

In [2]:
# Just rename silver_dim_date as gold_dim_date
df_date = spark.table('silver_dim_table')
df_date.write.format('delta').mode('overwrite').saveAsTable('gold_dim_date')

StatementMeta(, d3545ef4-c4c5-4271-8d74-7e82597a9333, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import functions as F

df_state = spark.table('silver_electric_vehicle_sales_by_state')
df_date = spark.table('silver_dim_table')

gold_state = df_state \
    .join(df_date, 'date', 'inner') \
    .groupBy('date', 'fiscal_year', 'quarter', 'state', 'vehicle_category') \
    .agg(
        F.sum('electric_vehicles_sold').alias('total_ev_sold'),
        F.sum('total_vehicles_sold').alias('total_vehicles'),
        F.round(
            F.sum('electric_vehicles_sold') / F.sum('total_vehicles_sold') * 100, 2
        ).alias('ev_penetration_rate')
    )

print(gold_state.count())

gold_state.write \
    .format('delta') \
    .mode('overwrite') \
    .option("mergeSchema", "true") \
    .saveAsTable('gold_ev_sales_by_state')

StatementMeta(, 88f45e28-03ab-4913-930b-032c0a88620b, 6, Finished, Available, Finished, False)

2445
